# RetailPulse: AI-Powered Sales & Retail Analytics Platform
## Day 7 — Week 1 Checkpoint

**Module:** MLOps — Experiment Tracking (Phase 1 wrap-up)

Per the execution plan, Day 7 is a consolidation checkpoint rather than new modeling. This notebook:
1. Verifies all Week 1 artifacts exist (raw data, cleaned data, features, EDA report)
2. Logs the Day 5 (Prophet) and Day 6 (LSTM) baseline models into MLflow — params, metrics, and artifacts — establishing the experiment-tracking practice the rest of the project will follow
3. Produces a short Week 1 status report

In [1]:
import os
import json
import pandas as pd
import mlflow

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
REPORTS_DIR = "../reports"
MODELS_DIR = "../models"

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("RetailPulse-Forecasting")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/07/18 08:46:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/18 08:46:42 INFO mlflow.store.db.utils: Updating database tables


2026/07/18 08:46:44 INFO mlflow.tracking.fluent: Experiment with name 'RetailPulse-Forecasting' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:///../mlflow.db


## 1. Artifact Inventory Check

In [2]:
expected_files = {
    "Raw data": [
        f"{RAW_DIR}/products_master.csv", f"{RAW_DIR}/customers_master.csv",
        f"{RAW_DIR}/sales_transactions.csv", f"{RAW_DIR}/inventory_snapshots.csv",
    ],
    "Cleaned / processed data": [
        f"{PROCESSED_DIR}/sales_cleaned.csv", f"{PROCESSED_DIR}/returns_cleaned.csv",
        f"{PROCESSED_DIR}/rfm_features.csv", f"{PROCESSED_DIR}/timeseries_daily.csv",
        f"{PROCESSED_DIR}/customer_segments.csv", f"{PROCESSED_DIR}/prophet_ready.csv",
        f"{PROCESSED_DIR}/timeseries_features_full.csv",
    ],
    "Baseline models": [
        f"{MODELS_DIR}/prophet_model.json", f"{MODELS_DIR}/prophet_metrics.json",
        f"{MODELS_DIR}/lstm_forecaster.pt", f"{MODELS_DIR}/lstm_metrics.json",
    ],
}

all_ok = True
for category, files in expected_files.items():
    print(f"\n{category}:")
    for f in files:
        exists = os.path.exists(f)
        all_ok = all_ok and exists
        print(f"  [{'OK' if exists else 'MISSING'}] {f}")

print(f"\nWeek 1 artifact check: {'ALL PRESENT' if all_ok else 'INCOMPLETE — see MISSING above'}")


Raw data:
  [OK] ../data/raw/products_master.csv
  [OK] ../data/raw/customers_master.csv
  [OK] ../data/raw/sales_transactions.csv
  [OK] ../data/raw/inventory_snapshots.csv

Cleaned / processed data:
  [OK] ../data/processed/sales_cleaned.csv
  [OK] ../data/processed/returns_cleaned.csv
  [OK] ../data/processed/rfm_features.csv
  [OK] ../data/processed/timeseries_daily.csv
  [OK] ../data/processed/customer_segments.csv
  [OK] ../data/processed/prophet_ready.csv
  [OK] ../data/processed/timeseries_features_full.csv

Baseline models:
  [OK] ../models/prophet_model.json
  [OK] ../models/prophet_metrics.json
  [OK] ../models/lstm_forecaster.pt
  [OK] ../models/lstm_metrics.json

Week 1 artifact check: ALL PRESENT


## 2. Log Baseline Models to MLflow

Each prior notebook (Day 5, Day 6) trained a model but only saved raw artifacts to disk. This is where those runs get formally registered — params, metrics, and artifacts — so future retraining (Day 13 automated pipeline) can compare against these baselines.

In [3]:
with open(f"{MODELS_DIR}/prophet_metrics.json") as f:
    prophet_metrics = json.load(f)

with mlflow.start_run(run_name="day5_prophet_baseline"):
    mlflow.log_params({
        "model_type": "Prophet",
        "seasonality_mode": "multiplicative",
        "changepoint_prior_scale": 0.05,
        "horizon_days": prophet_metrics["horizon_days"],
    })
    mlflow.log_metrics({
        "mape": prophet_metrics["mape"],
        "mae": prophet_metrics["mae"],
        "rmse": prophet_metrics["rmse"],
    })
    mlflow.log_artifact(f"{MODELS_DIR}/prophet_model.json")
    mlflow.log_artifact(f"{REPORTS_DIR}/day5_forecast_vs_actual.png")
    prophet_run_id = mlflow.active_run().info.run_id

print(f"Logged Prophet baseline. Run ID: {prophet_run_id}")

Logged Prophet baseline. Run ID: bfa80032a12d4e158d466a4f82149d65


In [4]:
with open(f"{MODELS_DIR}/lstm_metrics.json") as f:
    lstm_metrics = json.load(f)

with mlflow.start_run(run_name="day6_lstm_baseline"):
    mlflow.log_params({
        "model_type": "LSTM",
        "seq_len": lstm_metrics["seq_len"],
        "horizon_days": lstm_metrics["horizon_days"],
        "hidden_size": 64,
        "num_layers": 2,
    })
    mlflow.log_metrics({
        "mape": lstm_metrics["mape"],
        "mae": lstm_metrics["mae"],
        "rmse": lstm_metrics["rmse"],
    })
    mlflow.log_artifact(f"{MODELS_DIR}/lstm_forecaster.pt")
    mlflow.log_artifact(f"{REPORTS_DIR}/day6_lstm_forecast_vs_actual.png")
    lstm_run_id = mlflow.active_run().info.run_id

print(f"Logged LSTM baseline. Run ID: {lstm_run_id}")

Logged LSTM baseline. Run ID: 9af9746024c047dabd809b5a5afc9eeb


## 3. Compare Logged Runs

In [5]:
runs_df = mlflow.search_runs(experiment_names=["RetailPulse-Forecasting"])
cols = ["tags.mlflow.runName", "metrics.mape", "metrics.mae", "metrics.rmse"]
runs_df[cols].sort_values("metrics.mape")

,tags.mlflow.runName,metrics.mape,metrics.mae,metrics.rmse
0,day6_lstm_baseline,6.266,18200.55,22075.90
1,day5_prophet_baseline,8.515,23539.59,28775.61


## 4. Week 1 Status Report

In [6]:
sales_clean = pd.read_csv(f"{PROCESSED_DIR}/sales_cleaned.csv")
rfm = pd.read_csv(f"{PROCESSED_DIR}/customer_segments.csv")

summary = f"""
WEEK 1 CHECKPOINT — RetailPulse: AI-Powered Sales & Retail Analytics Platform
{'='*70}

Data Pipeline:
  - Raw transactions generated : {pd.read_csv(f'{RAW_DIR}/sales_transactions.csv').shape[0]:,} rows
  - Cleaned transactions       : {len(sales_clean):,} rows
  - Customers segmented        : {len(rfm):,}

Baseline Models (Day 5-6):
  - Prophet MAPE : {prophet_metrics['mape']:.2f}%
  - LSTM MAPE    : {lstm_metrics['mape']:.2f}%
  - Both models beat the <=12% MAPE target from the F-03 requirement.

Segmentation (Day 3):
  - {rfm['Segment'].nunique()} business segments identified via K-Means
  - DBSCAN cross-check applied for outlier detection

MLOps:
  - {len(runs_df)} runs logged in MLflow experiment 'RetailPulse-Forecasting'

Status: Week 1 objectives complete. Proceeding to Week 2 (hybrid ensemble,
churn prediction, inventory optimization).
"""

print(summary)

with open(f"{REPORTS_DIR}/week1_checkpoint_summary.txt", "w") as f:
    f.write(summary)


WEEK 1 CHECKPOINT — RetailPulse: AI-Powered Sales & Retail Analytics Platform

Data Pipeline:
  - Raw transactions generated : 471,418 rows
  - Cleaned transactions       : 420,663 rows
  - Customers segmented        : 4,955

Baseline Models (Day 5-6):
  - Prophet MAPE : 8.52%
  - LSTM MAPE    : 6.27%
  - Both models beat the <=12% MAPE target from the F-03 requirement.

Segmentation (Day 3):
  - 4 business segments identified via K-Means
  - DBSCAN cross-check applied for outlier detection

MLOps:
  - 2 runs logged in MLflow experiment 'RetailPulse-Forecasting'

Status: Week 1 objectives complete. Proceeding to Week 2 (hybrid ensemble,
churn prediction, inventory optimization).



## 5. Day 7 Checkpoint Summary

**Outputs saved:**
- `mlflow.db` — SQLite-backed MLflow tracking store with 2 logged baseline runs
- `reports/week1_checkpoint_summary.txt` — plain-text Week 1 status report

**Week 1 complete:** data pipeline (Days 1-2), segmentation (Day 3), forecasting data prep (Day 4), and two baseline forecasting models (Day 5-6) — all tracked in MLflow.

**Next module:** `08_hybrid_ensemble` — Week 2 begins, combining Prophet + LSTM.